# Submission Packaging and Runtime Validation

**Purpose.** Build a clean agent archive, execute the staged package with
the official simulator, and emit a traceable manifest.

**Decision question.** Does the exact tar.gz candidate import, locate its deck,
select legal actions, finish a game, and preserve the required root layout?

The previous packaging run verified structure but did not execute staged
`main.py`; that gap allowed a working-directory deck-path defect to escape.
This revision makes runtime smoke validation part of packaging itself.

## 1. Discover immutable inputs

Simulator files come only from the attached competition data. Reviewed
policy and deck files come from the private agent-source dataset. The
staging directory is recreated from scratch on every run.

In [ ]:
from collections import Counter
from pathlib import Path
import ast
import hashlib
import importlib.util
import json
import shutil
import sys
import time
import tarfile

import pandas as pd

PACKAGE_DIR = Path("/kaggle/working/submission_agent")
ARCHIVE = Path("/kaggle/working/submission.tar.gz")
MANIFEST_PATH = Path("/kaggle/working/submission_manifest.json")
MAX_DECISIONS = 10_000

sample = sorted(Path("/kaggle/input").rglob("sample_submission/main.py"))[0].parent
candidates = [Path("../agent"), Path("agent")]
candidates += [
    path.parent for path in sorted(Path("/kaggle/input").rglob("main.py"))
    if "sample_submission" not in path.parts and "cg" not in path.parts
]
repo_agent = next(
    (path for path in candidates
     if (path / "main.py").exists() and (path / "deck.csv").exists()),
    None,
)
if repo_agent is None:
    raise FileNotFoundError("Attach the private agent-source dataset.")
print(f"Official runtime: {sample}")
print(f"Reviewed agent: {repo_agent}")

## 2. Assemble and statically validate

Static checks reject syntax errors, missing entrypoints, malformed deck
length, incomplete SDK copies, and unexpected source locations before the
more expensive simulator test.

In [ ]:
if PACKAGE_DIR.exists():
    shutil.rmtree(PACKAGE_DIR)
PACKAGE_DIR.mkdir(parents=True)
shutil.copytree(sample / "cg", PACKAGE_DIR / "cg")
shutil.copy2(repo_agent / "main.py", PACKAGE_DIR / "main.py")
shutil.copy2(repo_agent / "deck.csv", PACKAGE_DIR / "deck.csv")

source = (PACKAGE_DIR / "main.py").read_text(encoding="utf-8-sig")
tree = ast.parse(source)
names = {node.name for node in ast.walk(tree) if isinstance(node, ast.FunctionDef)}
assert "agent" in names, "main.py must define agent(obs_dict)."

deck = [
    int(value) for value in (PACKAGE_DIR / "deck.csv").read_text().splitlines()
    if value.strip()
]
assert len(deck) == 60, f"Expected 60 cards, found {len(deck)}."
display(pd.Series(Counter(deck), name="copies").rename_axis("card_id").to_frame())

required = {
    "main.py", "deck.csv", "cg/__init__.py", "cg/api.py", "cg/game.py",
    "cg/sim.py", "cg/utils.py", "cg/cg.dll", "cg/libcg.so",
}
staged = {
    str(path.relative_to(PACKAGE_DIR)).replace("\\", "/")
    for path in PACKAGE_DIR.rglob("*") if path.is_file()
}
assert required <= staged, f"Missing required files: {sorted(required - staged)}"

## 3. Execute the staged package

Import `main.py` from staging, not from the source dataset, and run one full
legality-checked self-play game. This specifically verifies module-relative
deck discovery and the Linux shared library used by Kaggle.

In [ ]:
sys.path.insert(0, str(PACKAGE_DIR))
spec = importlib.util.spec_from_file_location("staged_agent", PACKAGE_DIR / "main.py")
staged_agent = importlib.util.module_from_spec(spec)
spec.loader.exec_module(staged_agent)

from cg.api import to_observation_class
from cg.game import battle_finish, battle_select, battle_start

staged_deck = staged_agent.read_deck_csv()
assert staged_deck == deck
started = time.perf_counter()
decisions = 0
try:
    obs_dict, start_data = battle_start(staged_deck, staged_deck)
    assert obs_dict is not None, {
        "error_player": start_data.errorPlayer,
        "error_type": start_data.errorType,
    }
    while decisions < MAX_DECISIONS:
        obs = to_observation_class(obs_dict)
        if obs.current is not None and obs.current.result != -1:
            runtime_smoke = {
                "status": "finished", "winner": int(obs.current.result),
                "turn": int(obs.current.turn), "decisions": decisions,
                "seconds": time.perf_counter() - started,
            }
            break
        action = staged_agent.agent(obs_dict)
        select = obs.select
        assert isinstance(action, list)
        assert len(action) == len(set(action))
        assert select.minCount <= len(action) <= select.maxCount
        assert all(isinstance(index, int) for index in action)
        assert all(0 <= index < len(select.option) for index in action)
        obs_dict = battle_select(action)
        decisions += 1
    else:
        raise RuntimeError("Staged package reached the decision limit.")
finally:
    battle_finish()
display(runtime_smoke)

for cache in PACKAGE_DIR.rglob("__pycache__"):
    shutil.rmtree(cache)
for compiled in PACKAGE_DIR.rglob("*.pyc"):
    compiled.unlink()

## 4. Hash, archive, and inspect

Source hashes link the ladder artifact to reviewed repository files. The
archive hash identifies the exact uploaded bytes. Tar members are relative
to staging so `main.py` cannot be hidden under an accidental parent folder.

In [ ]:
def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()

if ARCHIVE.exists():
    ARCHIVE.unlink()
with tarfile.open(ARCHIVE, "w:gz", format=tarfile.PAX_FORMAT) as archive:
    for path in sorted(PACKAGE_DIR.rglob("*")):
        if path.is_file():
            archive.add(
                path,
                arcname=path.relative_to(PACKAGE_DIR).as_posix(),
                recursive=False,
            )

with tarfile.open(ARCHIVE, "r:gz") as archive:
    file_members = [member for member in archive.getmembers() if member.isfile()]
    members = [member.name for member in file_members]
    for member in file_members:
        stream = archive.extractfile(member)
        assert stream is not None
        while stream.read(1 << 20):
            pass
assert set(members) == required, {
    "missing": sorted(required - set(members)),
    "unexpected": sorted(set(members) - required),
}
assert not any(member.startswith("submission_agent/") for member in members)

manifest = {
    "format": "tar.gz",
    "main_sha256": sha256(PACKAGE_DIR / "main.py"),
    "deck_sha256": sha256(PACKAGE_DIR / "deck.csv"),
    "archive_sha256": sha256(ARCHIVE),
    "archive_bytes": ARCHIVE.stat().st_size,
    "members": members,
    "runtime_smoke": runtime_smoke,
}
MANIFEST_PATH.write_text(json.dumps(manifest, indent=2))
display(manifest)
print(f"Archive: {ARCHIVE} ({ARCHIVE.stat().st_size / 1e6:.2f} MB)")

## 5. Submission gate

A complete packaging run now proves static structure and one staged runtime
path. Submission still requires comparative evidence, recorded hashes,
review of live rules, and an intentional decision about the latest two
tracked ladder slots. Kaggle's validation episode remains the final runtime
authority.